# 4. Population dynamics and phenotypic switching

Migration models often include birth, death and transitions
between phenotypes. These processes have different conservation
laws and therefore occupy different pipeline phases.

**Learning objectives**

- order birth/death, phenotype switching and reorientation;
- verify the complete-state conservation rule `s -> s'`;
- plot total population and phenotype fractions; and
- connect an explicit multi-phase model to go-or-grow dynamics.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import (
    BirthDeathSpec,
    InteractionPipelineSpec,
    PhenotypeSwitchSpec,
    ReorientationSpec,
    ReorientationTermSpec,
)
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder


## Conservation means sampling the full channel state

At one node, `s` is the complete `(species, channels)` state. A
particle-number-conserving phenotype interaction samples one
admissible state `s'` with the same total occupancy. It must not
perform independent channel writes that could collide, create or
remove particles.

We first isolate switching for one time step and turn propagation
off. This makes the invariant directly observable.


In [ ]:
switch_nodes = np.zeros((4, 4, 2, 5), dtype=bool)
switch_nodes[1, 1, 0, 0] = True
switch_nodes[1, 2, 0, 4] = True
switch_nodes[2, 1, 1, 2] = True

switch_only_spec = ModelSpec(
    description=Description(title="Atomic phenotype switch"),
    space=SpaceSpec(geometry="square", boundary="periodic"),
    state=StateSpec(nodes=switch_nodes, restchannels=1, n_species=2),
    time=TimeSpec(steps=1, seed=41),
    dynamics=InteractionPipelineSpec(
        operators=[
            PhenotypeSwitchSpec(
                name="phenotype_switch",
                parameters={"rates": [[0.0, 1.0], [0.0, 0.0]]},
            )
        ],
        propagation=False,
    ),
    analysis=AnalysisSpec(observers=[NodeRecorder(), DensityRecorder()]),
)

switch_only = run_model(switch_only_spec, showprogress=False)
before = switch_only.lgca.nodes_t[0]
after = switch_only.lgca.nodes_t[1]
assert before.sum() == after.sum()
print("particles by species before:", before.sum(axis=(0, 1, 3)))
print("particles by species after: ", after.sum(axis=(0, 1, 3)))
print("total particles conserved:", before.sum(), after.sum())


The forced switch changes species identity while preserving total
particle number. Random walk, alignment and chemotaxis in
volume-exclusion models use the same full-state principle for their
particle-conserving channel transitions.


## A complete sequential biological pipeline

Birth/death may change total population, phenotype switching
redistributes that population between species, reorientation
changes channels without changing particle number, and propagation
moves the selected channels. The order is visible below.


In [ ]:
population_spec = ModelSpec(
    description=Description(title="Growing and switching population"),
    space=SpaceSpec(
        geometry="square",
        dims=(12, 12),
        boundary="periodic",
    ),
    state=StateSpec(
        density=0.2,
        restchannels=1,
        n_species=2,
    ),
    time=TimeSpec(steps=20, seed=44),
    dynamics=InteractionPipelineSpec(
        operators=[
            BirthDeathSpec(
                name="birth_death",
                parameters={
                    "birth_rate": [0.03, 0.01],
                    "death_rate": [0.005, 0.005],
                },
            ),
            PhenotypeSwitchSpec(
                name="phenotype_switch",
                parameters={
                    "rates": [[0.0, 0.08], [0.03, 0.0]],
                },
            ),
            ReorientationSpec(
                terms=[ReorientationTermSpec(name="random_walk")],
            ),
        ],
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

population_result = run_model(population_spec, showprogress=False)
print(population_result.metadata["schedule"])


In [ ]:
species_populations = population_result.lgca.dens_t.sum(axis=(1, 2))
species_fractions = species_populations / species_populations.sum(axis=1, keepdims=True)
steps = population_result.lgca.n_steps

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
axes[0].plot(steps, population_result.lgca.n_t, color="black")
axes[0].set(xlabel="time step", ylabel="total population")
axes[1].plot(steps, species_fractions[:, 0], label="phenotype 0")
axes[1].plot(steps, species_fractions[:, 1], label="phenotype 1")
axes[1].set(xlabel="time step", ylabel="population fraction", ylim=(0, 1))
axes[1].legend()
plt.show()
plt.close(fig)


Total population changes because birth/death is present. Phenotype
fractions can change through both differential birth rates and the
switch matrix. Interpreting either mechanism requires a control in
which the other is removed.


## Go-or-grow as an application

In go-or-grow models, cells switch between a moving and a resting,
proliferative state. BioLGCA also provides this coupled mechanism
as a registered interaction. We still add it explicitly to a full
specification rather than loading an example.


In [ ]:
go_or_grow_spec = ModelSpec(
    description=Description(title="Go-or-grow population expansion"),
    space=SpaceSpec(geometry="hex", dims=(18, 18), boundary="periodic"),
    state=StateSpec(density=0.1, restchannels=6),
    time=TimeSpec(steps=12, seed=47),
    dynamics=InteractionPipelineSpec(
        operators=[
            {
                "name": "classical.go_or_grow",
                "parameters": {
                    "r_b": 0.2,
                    "r_d": 0.01,
                    "kappa": 4.0,
                    "theta": 0.75,
                },
            }
        ],
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

go_or_grow = run_model(go_or_grow_spec, showprogress=False)
print("initial and final population:", go_or_grow.lgca.n_t[[0, -1]])


## Exercises

1. Set the switch rates to zero. Which changes remain in phenotype
   fractions and why?
2. Give both species the same birth rate and isolate switching.
3. Reverse the two off-diagonal switch rates and predict the final
   phenotype balance before running the model.
4. For go-or-grow, record moving and resting populations separately
   and relate them to total growth.
